# sg-sem: build the data and train the model

The Skill Graph semantic checker (see `SKILL_GRAPH_SEMANTIC_MODEL_SPEC.md` in the repo). This notebook runs on a **Colab GPU** through the VS Code Colab extension:

1. clones the repo and runs our own graph library (Node) to build seed graphs and text jobs,
2. runs an open model **on this GPU** (no API) to write outlines, checker answers, plain-word phrases and gates,
3. assembles the labelled examples, split by graph,
4. fine-tunes the model and reports dev and test metrics against the 0.9 precision bar.
5. scores the hand-written **gold set** (`gold/`), which the model never saw: it decides what ships.

**Run it with `QUICK_LOOK = True` first** (about 15 minutes): it writes a few answers per round so their quality can be checked. Then set it to `False` and run all again; finished answers are kept, so nothing is redone.

Save the notebook when it finishes: its outputs are what gets reviewed.

In [13]:
REPO = "https://github.com/Saad-web-spec/DevScratchPad.git"
BRANCH = "sg-sem"  # the branch with the sg-sem code
GENERATOR = "Qwen/Qwen2.5-14B-Instruct"  # writes the text; runs here on the GPU (fp16 on an A100, 4-bit on smaller cards). 7B through msmarco-5: its H1 and H4 answers are kept on Drive and reused; H2 rewrites and the label checks are new
BASE = "cross-encoder/ms-marco-MiniLM-L12-v2"  # the encoder sg-sem is fine-tuned from: the same MiniLM-L12-H384 network, already trained on text pairs (the raw microsoft/MiniLM-L12-H384-uncased never was)
EPOCHS = 10                                 # 4 through minilm-5: H2 loss was still falling at the end
QUICK_LOOK = False                         # True: a few answers per round, to check quality first (done; the full data is on Drive)
TRAIN = not QUICK_LOOK                     # training waits for the full data
RUN = "msmarco-6"  # minilm-1: H1 missed buried problems, H2 at chance. minilm-2: H1 flagged honest number ledgers, H2 starved (68 positives). minilm-3: H2 read the input list, not the task (dropped/swapped negatives added). minilm-4: H2 only woke on an appended sentence (woven positives added). minilm-5: 9 in 10 woven answers unusable (bracketed mention, drafted twin); the raw encoder never learnt to match A with B, so the base is now the pair-trained checkpoint of the same network. msmarco-1: every full-list row was a negative, so H2 read the length of the input list (replaced positives added). msmarco-2: shortcut closed but 391 H2 positives were too few to learn the comparison (H2-only outlines, every input woven, H2 feature dropout). msmarco-3: 1,170 positives, gold AUROC 0.74, but 1 of 28 cuts at the threshold: the model never saw the step that was missing (H2 pairs now name a candidate step; no new text-model calls). msmarco-4: gold H2 AUROC 0.81, 13 of 28 at 0.5 but none above the dev threshold 0.915; the worst dev rows were positives whose marked words did not name the cut step (ungrounded answers dropped). msmarco-5: only 35 dropped, gold H2 4 of 28; the misses are cuts named as a person writes a task ("every extracted receipt"), which the 7B model never wrote, and H1 collapsed on honest answers that use the words no / none / only. Now: Qwen-14B, three mention styles, a label check over every H2 pair, honest H1 answers in problem-report words. See MEMORY.md

## 1. Storage and setup

In [14]:
import os, shutil, subprocess
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/sg-sem"
except Exception as error:  # the VS Code extension may not support Drive mounting
    print("Google Drive not mounted:", error)
    print("Data stays on this runtime only: if it disconnects, the run starts again.")
    ROOT = "/content/sg-sem"
DATA = f"{ROOT}/data-quick" if QUICK_LOOK else f"{ROOT}/data"
RUNS = f"{ROOT}/runs"
os.makedirs(DATA, exist_ok=True); os.makedirs(RUNS, exist_ok=True)
print("data:", DATA)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
data: /content/drive/MyDrive/sg-sem/data


In [15]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
CODE = "/content/DevScratchPad"
if not os.path.exists(CODE):
    !git clone --depth 1 -b {BRANCH} {REPO} {CODE}
else:
    !git -C {CODE} pull --ff-only
!git -C {CODE} log --oneline -1
node = subprocess.run(["node", "--version"], capture_output=True, text=True).stdout.strip() if shutil.which("node") else ""
if not node.startswith(("v20", "v22", "v24")):
    !curl -fsSL https://deb.nodesource.com/setup_22.x | bash - > /dev/null 2>&1 && apt-get install -y nodejs > /dev/null 2>&1
!node --version
%cd {CODE}
!npm ci --ignore-scripts --no-audit --no-fund --loglevel=error
!pip install -q -r ml/sg-sem/requirements.txt
PIPE = "npx vite-node scripts/semantic/pipeline.ts"
LIMIT = "--limit 48" if QUICK_LOOK else ""

name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB
remote: Enumerating objects: 45, done.
remote: Counting objects: 100% (45/45), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 26 (delta 21), reused 24 (delta 19), pack-reused 0 (from 0)
Unpacking objects: 100% (26/26), 31.32 KiB | 1.16 MiB/s, done.
From https://github.com/Saad-web-spec/DevScratchPad
   4137d21..a107025  sg-sem     -> origin/sg-sem
Updating 4137d21..a107025
Fast-forward
 MEMORY.md                                         |   8 +
 SKILL_GRAPH_SEMANTIC_MODEL_SPEC.md                |   8 +-
 ml/sg-sem/make_notebook.py                        |  13 +-
 ml/sg-sem/sg_sem.ipynb                            | 759 +++++++++++-----------
 ml/sg-sem/sgsem/generate.py                       |   8 +-
 scripts/semantic/llmPrompts.ts                    |  80 ++-
 scripts/semantic/pipeline.ts                      | 110 +++-
 src/app/graph/lib/__tests__/semanticInput.test.ts |  28 +-
 8 files changed, 618 insertion

## 2. Round 1: workflow outlines → seed graphs

In [16]:
!{PIPE} round1 --out {DATA}
!cd ml/sg-sem && python -m sgsem.generate --requests {DATA}/round1.requests.jsonl --outputs {DATA}/round1.outputs.jsonl --model {GENERATOR} {LIMIT}

⠙round1: 520 outline requests → /content/drive/MyDrive/sg-sem/data/round1.requests.jsonl
⠙520 already answered, 0 to go


## 3. Round 2: checker answers, plain-word inputs, gates

In [17]:
!{PIPE} round2 --out {DATA}
!cd ml/sg-sem && python -m sgsem.generate --requests {DATA}/round2.requests.jsonl --outputs {DATA}/round2.outputs.jsonl --model {GENERATOR} {LIMIT}

⠙round2: 437 seeds (423 drafted; rejected {"json":3,"draft":94,"errors":0,"duplicate":0}), 12070 requests {"h1":9555,"h4":455,"h2":2060}
why outlines were rejected:
  46 × steps[N].type must be one of orchestrator, worker, verifier, synthesizer, agent, code, approval.
  12 × steps[N].foreach must be <step> or <step>.<key>, e.g. plan.files: the step whose JSON list this step runs over.
  11 × Step … fans out over …, so … must be in its after.
  3 × steps[N].id must start with a letter and use only letters, digits, _ or - (got …).
  3 × Step … reads from …, which is not a step. Use the ids of other steps, or leave after empty to read the request.
  1 × Step … fans out over …, but … has no … among its keys (laptopN-summary, laptopN-summary, laptopN-summary).
  1 × Step … fans out over …, but … has no … among its keys (length, tone).
  1 × Step … fans out over …, but … has no … among its keys (story).
⠙8700 already answered, 3370 to go
tokenizer_config.json: 7.30kB [00:00, 30.6MB/s]
vocab.

## 4. Assemble the labelled examples, then check every H2 pair's label

In [1]:
!{PIPE} assemble --out {DATA}
print(open(f"{DATA}/stats.json").read())

/bin/bash: line 1: {PIPE}: command not found


NameError: name 'DATA' is not defined

In [19]:
# Round 3: the text model reads every version of a step once (task, connected inputs, unconnected steps) and names the
# unconnected steps the task works on. Positives it does not confirm and negatives it names are dropped (judge).
!{PIPE} round3 --out {DATA}
!cd ml/sg-sem && python -m sgsem.generate --requests {DATA}/round3.requests.jsonl --outputs {DATA}/round3.outputs.jsonl --model {GENERATOR} {LIMIT}
!{PIPE} judge --out {DATA}

⠙round3: 6561 label checks over 12763 H2 pairs
⠙0 already answered, 6561 to go
`torch_dtype` is deprecated! Use `dtype` instead!
I0000 00:00:1792760922.053790   21189 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792760922.127274   21189 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792760924.553078   21189 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPT

In [20]:
# For review: raw answers of the text model, two per kind and style (newest first), and the assembled samples.
import json, collections
requests = {json.loads(l)["id"]: json.loads(l) for l in open(f"{DATA}/round2.requests.jsonl")}
shown = collections.Counter()
for line in reversed(open(f"{DATA}/round2.outputs.jsonl").readlines()):
    row = json.loads(line); request = requests.get(row["id"])
    if request is None:  # an answer to an older prompt
        continue
    kind = (request["kind"], request["meta"].get("style", request["meta"].get("mode", "")))
    if shown[kind] < 2:
        shown[kind] += 1
        print(f"--- {' '.join(filter(None, kind))} {request['meta']}")
        print(row["text"][:1200])
        print()
print(open(f"{DATA}/samples.md").read()[:12000])

--- h2 shape {'ref': 'CHECK EXPERIENCE', 'mode': 'weave', 'style': 'shape', 'seed': 'draft-h2-outline-3-0', 'node': 'generate_report', 'pred': 'check_experience'}
{"task": "Summarize the cover letter's fit using [the JSON with key 'experience'], the skills check, and the style assessment: recommend encouragement or criticism.", "task_without": "Summarize the cover letter's fit using the skills check and the style assessment: recommend encouragement or criticism."}

--- h2 origin {'ref': 'VERIFY REPORTS', 'mode': 'weave', 'style': 'origin', 'seed': 'draft-h2-outline-3-0', 'node': 'generate_report', 'pred': 'verify_reports'}
{"task": "Generate a comprehensive report on the cover letter from the [verified] experience, skills, and style assessments, including recommendations and specific feedback.", "task_without": "Generate a comprehensive report on the cover letter from the experience, skills, and style assessments, including recommendations and specific feedback."}

--- h2 origin {'ref'

## 5. Train (runs when `QUICK_LOOK = False`)

In [21]:
if TRAIN:
    !cd ml/sg-sem && python -m sgsem.train --data {DATA}/examples.jsonl --out {RUNS}/{RUN} --base {BASE} --epochs {EPOCHS}
else:
    print("QUICK_LOOK is on: training waits for the full data.")

train examples: {'h1': 2190, 'h2': 5956, 'h3': 733, 'h4': 2792} dev: {'h1': 378, 'h2': 1108, 'h3': 102, 'h4': 424}
I0000 00:00:1792761591.208224   24185 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792761591.280788   24185 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792761593.518636   24185 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00

In [22]:
import json
if TRAIN:
    report = json.load(open(f"{RUNS}/{RUN}/metrics.json"))
    print(json.dumps({k: report[k] for k in ["test", "rules_baseline_test", "meets_bar_on_test", "minutes"]}, indent=1))
    for section in open(f"{RUNS}/{RUN}/errors.md").read().split("\n## ")[1:]:  # the worst dev examples, per head
        print("## " + section[:2500])

{
 "test": {
  "h1": {
   "n": 246,
   "temperature": 2.5,
   "precision_at_05": 0.9869281045751634,
   "recall_at_05": 0.9869281045751634,
   "f1_at_05": 0.9869281045751634,
   "auroc": 0.9953615854944128,
   "threshold": 0.5,
   "precision_at_threshold": 0.9869281045751634,
   "recall_at_threshold": 0.9869281045751634,
   "score": 0.9869281045751634,
   "sentence_f1": 0.8425925925925924
  },
  "h2": {
   "n": 496,
   "temperature": 2.9000000000000004,
   "pairs": 880,
   "pair_auroc": 0.9795295613477432,
   "right_candidate": 0.9818181818181818,
   "precision_at_05": 0.8603351955307262,
   "recall_at_05": 0.9333333333333333,
   "f1_at_05": 0.8953488372093025,
   "auroc": 0.9737251670786414,
   "threshold": 0.9359999895095825,
   "precision_at_threshold": 0.948905109489051,
   "recall_at_threshold": 0.7878787878787878,
   "score": 0.8953488372093025,
   "span_f1": 0.9351485144579774
  },
  "h3": {
   "n": 73,
   "temperature": 4.7,
   "precision_at_05": 0.8928571428571429,
   "recall_

## 6. Gold set (decides what ships)

Hand-written graphs, checker answers and gates in `ml/sg-sem/gold/`, on topics outside the training topics and never touched by the text model. Scored with the run's own thresholds, so the numbers are what the connector would do. A head ships only at precision >= 0.9 here.

In [23]:
import json
!npx vite-node scripts/semantic/gold.ts
!cd ml/sg-sem && python -m sgsem.evaluate --run {RUNS}/{RUN} --data gold/examples.jsonl --out {RUNS}/{RUN}/gold
gold = json.load(open(f"{RUNS}/{RUN}/gold/metrics.json"))
print(json.dumps({k: gold[k] for k in ["metrics", "rules_baseline", "meets_bar"]}, indent=1))
for section in open(f"{RUNS}/{RUN}/gold/errors.md").read().split("\n## ")[1:]:  # the worst gold examples, per head
    print("## " + section[:3500])

⠙gold: 7 graphs, 173 examples → ml/sg-sem/gold/examples.jsonl
{"h1":{"label0":20,"label1":22},"h2":{"label0":40,"label1":28},"h4":{"label0":28,"label1":14,"label2":21}}
⠙WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
I0000 00:00:1792761831.179624   25366 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792761831.254963   25366 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792761833.653622   25366 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floatin

In [24]:
# Which cue does H2 read? Every gold cut positive is scored as is and with one thing changed; the column whose p(cut) moves is the cue.
!cd ml/sg-sem && python -m sgsem.probe --run {RUNS}/{RUN} --data gold/examples.jsonl

h2: 68 gold rows, 28 positives
I0000 00:00:1792761849.654404   25496 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792761849.722630   25496 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792761852.108113   25496 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792761852.108607   25496 cudart_stub.cc:31] Could not find cuda drivers on your

In [25]:
# Does one side alone give the H2 label away on the generated rows? Classifiers on text A only, text B only, the 16 facts only; test AUROC per negative kind.
!cd ml/sg-sem && python -m sgsem.leak --data {DATA}/examples.jsonl

h2: 9740 rows
label kind          rows  no {{ref}} in A  blocks in A  lines in B  A words
0     as written    1600             0.00         1.04        1.31       33
0     drafted       1105             0.00         1.70        1.70       33
0     dropped        737             0.00         1.00        1.36       26
0     kept          1245             0.09         0.60        1.61       32
0     other         3182             0.04         0.79        1.24       30
1     cut           1125             0.01         0.92        1.30       31
1     replaced       746             0.00         1.95        1.95       34
rows whose A has no reference at all: 230, of them positive: 0.03

test AUROC of a train-split classifier that sees one side only (0.5 = the side carries nothing, near 1 = the label is in that side alone):
all positives             vs kept     vs drafted     vs dropped  vs as written         vs all
text A only                  0.58           0.57           0.65           0.78